# **Variasi Bahasa Lokal Terkontrol | MASSIVE-Agents**

---

**Project:** KP Agentic AI + LLM  
**Tahap:** Variasi bahasa lokal (Step 9)  
**Dataset:** `id-ID` · `development`

Variasi dibuat dari instruksi Indonesia yang sama dengan lima kategori: slang, singkatan, typo, code-switching ID–EN, dan ID–Jawa. Ground truth, `group_id`, serta split asal tidak diubah. Semua hasil masih berupa kandidat yang perlu validasi manual.

## **1 · Dataset dan Konfigurasi**

---

Dataset berasal dari hasil preprocessing yang telah direview. Mode `pilot_40` memilih maksimal 40 kandidat instruksi dengan aturan transformasi yang tersedia. Mode `development_full` menggunakan semua kandidat yang memenuhi syarat otomatis. Keduanya hanya memakai `development`.

In [1]:
from __future__ import annotations

import csv
import hashlib
import json
import os
import random
import re
from collections import Counter
from datetime import datetime, timezone
from pathlib import Path

from IPython.display import display
import pandas as pd

RUN_MODE = "pilot_40"  # pilot_40 / development_full
SEED = 20261110
JUMLAH_PILOT = 40
JUMLAH_AUDIT_PER_KATEGORI = 20 if RUN_MODE == "pilot_40" else 100
PROJECT_ROOT_OVERRIDE = None

CATEGORIES = ["slang", "singkatan", "typo", "code_switch_id_en", "code_switch_id_jawa"]


def project_root():
    env_root = os.environ.get("KP_AGENTIC_AI_ROOT")
    candidates = []
    if PROJECT_ROOT_OVERRIDE:
        candidates.append(Path(PROJECT_ROOT_OVERRIDE))
    if env_root:
        candidates.append(Path(env_root))
    cwd = Path.cwd().resolve()
    candidates.extend([cwd, *cwd.parents, Path(r"D:\KP-Agentic-AI")])
    for root in candidates:
        if (root / "data" / "processed" / "massive_agents_reviewed" / "id-ID_development.jsonl").is_file():
            return root
    raise FileNotFoundError("File id-ID_development.jsonl belum ditemukan di data/processed/massive_agents_reviewed")

ROOT = project_root()
INPUT = ROOT / "data" / "processed" / "massive_agents_reviewed" / "id-ID_development.jsonl"
OUTPUT = ROOT / "results" / "variasi_bahasa_terkontrol" / RUN_MODE
OUTPUT.mkdir(parents=True, exist_ok=True)

print("Dataset:", INPUT)
print("Mode:", RUN_MODE)
print("Hasil:", OUTPUT)

Dataset: D:\KP-Agentic-AI\data\processed\massive_agents_reviewed\id-ID_development.jsonl
Mode: pilot_40
Hasil: D:\KP-Agentic-AI\results\variasi_bahasa_terkontrol\pilot_40


## **2 · Kandidat Instruksi Formal**

---

Penyaringan awal menggunakan kata-kata informal yang mudah dikenali. Hasilnya hanya **kandidat formal otomatis**, bukan bukti bahwa kalimat benar-benar formal atau GT sudah tepat. Instruksi yang sama hanya diambil satu kali untuk mengurangi pengulangan.

In [2]:
with INPUT.open(encoding="utf-8") as f:
    records = [json.loads(line) for line in f if line.strip()]

assert len(records) == 397, f"Jumlah id-ID development berubah: {len(records)}"
assert all(r.get("language_code") == "id-ID" and r.get("split") == "development" for r in records)
assert len({r["row_key"] for r in records}) == len(records)

KATA_TIDAK_FORMAL = re.compile(
    r"\b(olly|siri|alexa|hey|ayo|aku|gak|nggak|ngga|pengen|dong|ngelawak|gue|gw|lo|lu)\b",
    flags=re.IGNORECASE
)

def kandidat_formal(question):
    return bool(question.strip()) and not KATA_TIDAK_FORMAL.search(question)

raw_formal = [r for r in records if kandidat_formal(r["question"])]
unique_formal = []
seen_text = set()
for record in raw_formal:
    norm = " ".join(record["question"].casefold().split())
    if norm not in seen_text:
        seen_text.add(norm)
        unique_formal.append(record)

print("Data ID development:", len(records))
print("Kandidat formal awal:", len(raw_formal))
print("Kandidat formal setelah deduplikasi teks:", len(unique_formal))

Data ID development: 397
Kandidat formal awal: 365
Kandidat formal setelah deduplikasi teks: 359


## **3 · Aturan Transformasi**

---

Slang, singkatan, ID–EN, dan ID–Jawa memakai daftar penggantian terbatas. Typo memakai **satu perubahan huruf yang sudah ditetapkan**. Kata yang muncul sebagai nilai argumen GT dihindari agar nilai entitas, lokasi, dan waktu tidak diubah secara tidak sengaja.

Aturan tidak menghasilkan parafrase bebas dan tidak memakai model generator. Kosakata Jawa masih perlu diperiksa oleh penutur yang memahami ragam bahasa yang dipakai.

In [3]:
RULES = {
    "slang": [
        ("beri tahu", "kasih tau"), ("beritahu", "kasih tau"),
        ("bagaimana", "gimana"), ("tidak", "nggak"), ("ingin", "pengen"),
        ("tolong", "tolong dong"), ("saya", "aku"), ("apakah", "apa"),
        ("mainkan", "putarin"), ("putarkan", "putarin"),
    ],
    "singkatan": [
        ("bagaimana", "gmn"), ("tolong", "tlg"), ("sekarang", "skrg"),
        ("berapa", "brp"), ("dengan", "dgn"), ("untuk", "utk"),
        ("saya", "sy"), ("yang", "yg"), ("apakah", "apkh"),
    ],
    "typo": [
        ("bagaimana", "bagaimna"), ("tolong", "tolnog"),
        ("apakah", "apkaah"), ("sekarang", "sekarnag"),
        ("berapa", "berpa"), ("dengan", "dnegan"),
        ("untuk", "utnuk"), ("saya", "syaa"), ("yang", "yagn"),
    ],
    "code_switch_id_en": [
        ("tolong", "please"), ("cuaca", "weather"),
        ("berita", "news"), ("musik", "music"),
        ("mainkan", "play"), ("putar", "play"),
        ("hapus", "delete"), ("nyalakan", "turn on"),
        ("matikan", "turn off"), ("waktu", "time"),
    ],
    "code_switch_id_jawa": [
        ("tolong", "tulung"), ("berapa", "pira"),
        ("sekarang", "saiki"), ("bagaimana", "piye"),
        ("jangan", "ojo"), ("mau", "arep"),
        ("untuk", "kanggo"), ("dengan", "karo"),
        ("apakah", "opo"), ("apa", "opo"), ("hari ini", "dina iki"),
        ("besok", "sesuk"),
    ],
}

# Setiap aturan hanya mengganti satu frasa per instruksi.
def pattern_for(term):
    return re.compile(r"(?<!\w)" + re.escape(term) + r"(?!\w)", re.IGNORECASE)


def all_gt_values(value):
    if isinstance(value, dict):
        for v in value.values():
            yield from all_gt_values(v)
    elif isinstance(value, (list, tuple)):
        for v in value:
            yield from all_gt_values(v)
    elif isinstance(value, str) and value.strip():
        yield value


def gt_contains(record, term):
    p = pattern_for(term)
    return any(p.search(value) for value in all_gt_values(record["gt"]))


def make_variant(record, category):
    original = record["question"]
    for old, new in RULES[category]:
        match = pattern_for(old).search(original)
        if not match or gt_contains(record, old):
            continue
        changed = original[:match.start()] + new + original[match.end():]
        if changed != original:
            return changed, old, new
    return None

assert set(RULES) == set(CATEGORIES)
for category, rules in RULES.items():
    assert len(rules) == len(set(old for old, _ in rules))
print("Kategori aturan:", len(RULES))

Kategori aturan: 5


## **4 · Pemilihan Induk dan Pembentukan Variasi**

---

Pemilihan pilot mengutamakan instruksi yang bisa menghasilkan lebih banyak kategori, lalu menjaga keragaman fungsi target. Setiap varian menyimpan instruksi asal, fungsi target, GT, `group_id`, dan split yang sama. Kategori yang tidak memiliki aturan cocok tidak dipaksakan.

In [4]:
def available_categories(record):
    return [cat for cat in CATEGORIES if make_variant(record, cat) is not None]

eligible = [(r, available_categories(r)) for r in unique_formal]
eligible = [(r, cats) for r, cats in eligible if cats]

rng = random.Random(SEED)
ranked = sorted(
    eligible,
    key=lambda item: (-len(item[1]), hashlib.sha256((str(SEED) + item[0]["row_key"]).encode()).hexdigest()),
)

if RUN_MODE == "pilot_40":
    selected = []
    selected_ids = set()
    used_functions = set()
    for rec, cats in ranked:
        fn = next(iter(rec["gt"]))
        if fn not in used_functions and len(selected) < JUMLAH_PILOT:
            selected.append((rec, cats))
            selected_ids.add(rec["row_key"])
            used_functions.add(fn)
    for rec, cats in ranked:
        if len(selected) >= JUMLAH_PILOT:
            break
        if rec["row_key"] not in selected_ids:
            selected.append((rec, cats))
            selected_ids.add(rec["row_key"])
elif RUN_MODE == "development_full":
    selected = ranked
else:
    raise ValueError("RUN_MODE harus pilot_40 atau development_full")

parents, variants, missing = [], [], []
for record, cats in selected:
    function_name = next(iter(record["gt"]))
    gt_text = json.dumps(record["gt"], ensure_ascii=False, sort_keys=True)
    parents.append({
        "row_key_induk": record["row_key"], "source_record_id": record["source_record_id"],
        "group_id": record["group_id"], "split": record["split"],
        "instruksi_id_asli": record["question"], "fungsi_gt": function_name,
        "gt_json": gt_text, "status_formal_manual": "BELUM_DIREVIEW",
        "catatan_formal": "",
    })
    for category in CATEGORIES:
        made = make_variant(record, category)
        if made is None:
            missing.append({"row_key_induk": record["row_key"], "kategori": category, "alasan": "tidak ada aturan yang aman/cocok"})
            continue
        new_question, original_term, replacement = made
        variants.append({
            "variant_id": record["row_key"] + "::" + category,
            "row_key_induk": record["row_key"],
            "source_record_id": record["source_record_id"],
            "group_id": record["group_id"], "split": record["split"],
            "language_code": record["language_code"],
            "kategori": category,
            "instruksi_id_asli": record["question"],
            "instruksi_variasi": new_question,
            "aturan_asal": original_term, "aturan_pengganti": replacement,
            "fungsi_gt": function_name, "gt_json": gt_text,
            "status_validasi": "BELUM_DIREVIEW", "catatan": "",
        })

variants_df = pd.DataFrame(variants)
parent_df = pd.DataFrame(parents)
missing_df = pd.DataFrame(missing, columns=["row_key_induk", "kategori", "alasan"])
print("Induk terpilih:", len(parent_df))
print("Varian terbentuk:", len(variants_df))
print("Kategori tidak terbentuk:", len(missing_df))
display(variants_df[["kategori", "instruksi_id_asli", "instruksi_variasi"]].head(12))

Induk terpilih: 40
Varian terbentuk: 170
Kategori tidak terbentuk: 30


,kategori,instruksi_id_asli,instruksi_variasi
0,slang,tolong cek email baru,tolong dong cek email baru
1,singkatan,tolong cek email baru,tlg cek email baru
2,typo,tolong cek email baru,tolnog cek email baru
3,code_switch_id_en,tolong cek email baru,please cek email baru
4,code_switch_id_jawa,tolong cek email baru,tulung cek email baru
5,slang,tolong mainkan hanya musik jenis pop mix saja,tolong dong mainkan hanya musik jenis pop mix ...
6,singkatan,tolong mainkan hanya musik jenis pop mix saja,tlg mainkan hanya musik jenis pop mix saja
7,typo,tolong mainkan hanya musik jenis pop mix saja,tolnog mainkan hanya musik jenis pop mix saja
8,code_switch_id_en,tolong mainkan hanya musik jenis pop mix saja,please mainkan hanya musik jenis pop mix saja
9,code_switch_id_jawa,tolong mainkan hanya musik jenis pop mix saja,tulung mainkan hanya musik jenis pop mix saja


## **5 · Pemeriksaan Konsistensi**

---

Pemeriksaan otomatis memastikan setiap varian masih memiliki fungsi, GT, `group_id`, dan split yang sama dengan induknya. Pemeriksaan ini **tidak dapat membuktikan kesetaraan makna**. Validasi manual tetap dibutuhkan sebelum data digunakan untuk evaluasi model.

In [5]:
source_by_key = {r["row_key"]: r for r in records}
assert len(variants_df) > 0, "Tidak ada variasi yang terbentuk"
assert variants_df["variant_id"].is_unique
assert parent_df["row_key_induk"].is_unique
assert all(r["split"] == "development" for r in variants)
assert not any(r["instruksi_id_asli"] == r["instruksi_variasi"] for r in variants)

checks = []
for row in variants:
    original = source_by_key[row["row_key_induk"]]
    checks.append({
        "variant_id": row["variant_id"],
        "gt_sama": row["gt_json"] == json.dumps(original["gt"], ensure_ascii=False, sort_keys=True),
        "fungsi_sama": row["fungsi_gt"] == next(iter(original["gt"])),
        "group_id_sama": row["group_id"] == original["group_id"],
        "split_sama": row["split"] == original["split"] == "development",
        "sumber_teks_sama": row["instruksi_id_asli"] == original["question"],
    })
checks_df = pd.DataFrame(checks)
assert checks_df.drop(columns="variant_id").all().all(), "Ada ketidaksesuaian dengan dataset sumber"
assert len(set(parent_df["group_id"])) == len(parent_df), "Group ID induk terduplikasi"
print("Pemeriksaan GT, fungsi, grup, dan split: PASS")
print("Catatan: kesetaraan makna belum tervalidasi")

Pemeriksaan GT, fungsi, grup, dan split: PASS
Catatan: kesetaraan makna belum tervalidasi


## **6 · Sampel Validasi Manual**

---

Sampel audit dipilih secara acak dengan seed tetap pada setiap kategori. Pilot memakai maksimal 20 sampel per kategori, sedangkan mode penuh menyiapkan maksimal 100 sampel per kategori. Sampel ini tidak digunakan untuk menghitung persentase kesalahan seluruh dataset.

In [6]:
audit_parts = []
for category in CATEGORIES:
    cat_df = variants_df[variants_df["kategori"] == category]
    if not cat_df.empty:
        n = min(JUMLAH_AUDIT_PER_KATEGORI, len(cat_df))
        selected_audit = cat_df.sample(n=n, random_state=SEED).copy()
        audit_parts.append(selected_audit)

audit = pd.concat(audit_parts, ignore_index=True) if audit_parts else variants_df.iloc[:0].copy()
audit["induk_formal"] = "BELUM_DIREVIEW"
audit["makna_tetap"] = "BELUM_DIREVIEW"
audit["entitas_waktu_tetap"] = "BELUM_DIREVIEW"
audit["fungsi_gt_tetap"] = "BELUM_DIREVIEW"
audit["nilai_argumen_masih_valid"] = "BELUM_DIREVIEW"
audit["teks_wajar"] = "BELUM_DIREVIEW"
audit["keputusan_final"] = "BELUM_DIREVIEW"
audit["alasan_keputusan"] = ""

summary_rows = []
for category in CATEGORIES:
    count = int((variants_df["kategori"] == category).sum())
    summary_rows.append({
        "kategori": category,
        "jumlah_varian": count,
        "jumlah_sampel_audit": int((audit["kategori"] == category).sum()),
        "target_audit": JUMLAH_AUDIT_PER_KATEGORI,
        "memenuhi_target_audit": count >= JUMLAH_AUDIT_PER_KATEGORI,
    })
summary_df = pd.DataFrame(summary_rows)
display(summary_df)
print("Seluruh keputusan manual:", audit["keputusan_final"].unique().tolist())

,kategori,jumlah_varian,jumlah_sampel_audit,target_audit,memenuhi_target_audit
0,slang,35,20,20,True
1,singkatan,40,20,20,True
2,typo,40,20,20,True
3,code_switch_id_en,23,20,20,True
4,code_switch_id_jawa,32,20,20,True


Seluruh keputusan manual: ['BELUM_DIREVIEW']


## **7 · Penyimpanan Hasil**

---

File keluaran terdiri dari daftar induk, seluruh varian kandidat, sampel audit manual, kategori tanpa transformasi, pemeriksaan konsistensi, serta metadata percobaan. Dataset sumber dan ground truth asli tidak diubah.

In [7]:
def save_csv(df, name):
    path = OUTPUT / name
    df.to_csv(path, index=False, encoding="utf-8-sig")
    return path

out_files = [
    save_csv(parent_df, "kandidat_induk_formal.csv"),
    save_csv(variants_df, "variasi_bahasa_kandidat.csv"),
    save_csv(audit, "sampel_validasi_manual.csv"),
    save_csv(missing_df, "kategori_tidak_terbentuk.csv"),
    save_csv(checks_df, "pemeriksaan_konsistensi.csv"),
    save_csv(summary_df, "ringkasan_variansi.csv"),
]

metadata = {
    "run": "variasi_bahasa_terkontrol_step9_v1",
    "created_at_utc": datetime.now(timezone.utc).isoformat(),
    "run_mode": RUN_MODE, "seed": SEED,
    "input": str(INPUT),
    "input_sha256": hashlib.sha256(INPUT.read_bytes()).hexdigest(),
    "source_development_id": len(records),
    "candidate_formal_auto_unique": len(unique_formal),
    "selected_parents": len(parent_df),
    "variants": len(variants_df),
    "categories": CATEGORIES,
    "semantic_validation": "pending_manual_review",
    "gt_modified": False, "split": "development",
    "model_generator_used": False,
    "ready_for_final_model_evaluation": False,
    "notes": "Formalitas induk, dialek Jawa, kesetaraan semantik, dan GT harus diaudit manual sebelum evaluasi.",
}
metadata_file = OUTPUT / "metadata_variasi.json"
metadata_file.write_text(json.dumps(metadata, indent=2, ensure_ascii=False), encoding="utf-8")
out_files.append(metadata_file)
for path in out_files:
    print(path.name)

kandidat_induk_formal.csv
variasi_bahasa_kandidat.csv
sampel_validasi_manual.csv
kategori_tidak_terbentuk.csv
pemeriksaan_konsistensi.csv
ringkasan_variansi.csv
metadata_variasi.json


## **8 · Kesimpulan Sementara**

---

Hasil Step 9 adalah **kandidat variasi bahasa**, bukan dataset sintetis tervalidasi. Penggunaan untuk evaluasi model menunggu pemeriksaan formalitas instruksi induk, kestabilan maksud dan argumen, kelayakan bahasa Jawa, serta keputusan GT yang masih provisional.

Varian dengan keputusan `TIDAK_VALID` atau `RAGU` tidak digunakan sebagai data evaluasi akhir. Variasi dari record test hanya dibuat setelah aturan dan protokol validasi ditetapkan berdasarkan development.